<a href="https://colab.research.google.com/github/omkar-droid/tensorrt-inference-optimization/blob/main/notebooks/tensorrt_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TensorRT Inference Optimization with FP16 + INT8

End-to-end driver notebook: **PyTorch → ONNX → TensorRT engines → benchmarks → accuracy validation**.

Designed for **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4).  
T4 has compute capability 7.5 → both FP16 and INT8 Tensor Cores.

Total runtime end-to-end: ~30–45 minutes.

## 1. Verify GPU

In [1]:
!nvidia-smi

Fri May  8 01:26:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

Colab ships with PyTorch + CUDA. We add TensorRT, pycuda, ONNX, and polygraphy.

In [ ]:
!pip install -q tensorrt pycuda onnx onnxruntime-gpu polygraphy datasets matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.3/271.3 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.8/372.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.2 MB

In [ ]:
import tensorrt as trt
import torch
import pycuda.driver as cuda
cuda.init()
dev = cuda.Device(0)
print(f'TensorRT: {trt.__version__}')
print(f'PyTorch:  {torch.__version__} (CUDA {torch.version.cuda})')
print(f'GPU:      {dev.name()}  compute {dev.compute_capability()}')
assert dev.compute_capability() >= (7, 0), 'Need compute capability >= 7.0 for Tensor Cores'

## 3. Get the project source

Replace the URL with your fork once pushed. Until then, you can upload the `src/` folder to the Colab session manually.

In [ ]:
# Option A: clone from GitHub (replace with your repo URL)
# !git clone https://github.com/<your-user>/tensorrt-inference-optimization.git
# %cd tensorrt-inference-optimization

# Option B: if you uploaded src/ to /content, just %cd into it
%cd /content
!ls

## 4. Export PyTorch → ONNX

In [ ]:
!python -m src.export_onnx --model resnet18 --output engines/resnet18.onnx
!python -m src.export_onnx --model resnet34 --output engines/resnet34.onnx

## 5. Download ImageNet calibration + validation subsets

Calibration: 512 images for INT8 PTQ.  
Validation: 5,000 images for top-1/top-5 evaluation.

Note: HuggingFace `imagenet-1k` requires an HF account + accepting the dataset license. Run `huggingface-cli login` first if it fails.

In [ ]:
from pathlib import Path
from datasets import load_dataset
from tqdm import tqdm

def cache_subset(split, n, out_dir):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    ds = load_dataset('imagenet-1k', split=split, streaming=True)
    labels = []
    for i, ex in enumerate(tqdm(ds, total=n)):
        if i >= n: break
        p = out / f'img_{i:05d}.jpg'
        if not p.exists():
            ex['image'].convert('RGB').save(p, 'JPEG')
        labels.append(int(ex['label']))
    (out / 'labels.txt').write_text('\n'.join(map(str, labels)))
    return out

cache_subset('validation', 512, 'data/calibration')
cache_subset('validation', 5000, 'data/val_subset')

## 6. Build TensorRT engines (FP32 / FP16 / INT8) for both models

In [ ]:
for model in ['resnet18', 'resnet34']:
    for prec in ['fp32', 'fp16']:
        !python -m src.build_engine --onnx engines/{model}.onnx \
            --output engines/{model}_{prec}.engine --precision {prec}
    !python -m src.build_engine --onnx engines/{model}.onnx \
        --output engines/{model}_int8.engine --precision int8 \
        --calib-dir data/calibration \
        --calib-cache engines/{model}_calib.cache

In [ ]:
!ls -lh engines/*.engine

## 7. Smoke test: run inference on a sample image

In [ ]:
import urllib.request
urllib.request.urlretrieve(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg',
    'sample_cat.jpg'
)
!python -m src.infer --engine engines/resnet18_fp16.engine --image sample_cat.jpg

## 8. Benchmark all configurations

Runs 5 backends × 2 models × 3 batch sizes = 30 measurements with 50 warmup + 200 timed iterations each.

In [ ]:
!python -m src.benchmark --output results/benchmarks.json

## 9. Validate accuracy on 5K ImageNet val

The headline check: TRT FP16 within 0.1% and TRT INT8 within 1.0% of the PyTorch FP32 baseline.

In [ ]:
!python -m src.validate --output results/accuracy.json --num-samples 5000

## 10. Generate charts

In [ ]:
!python -m src.plot_results

In [ ]:
from IPython.display import Image, display
display(Image('results/latency_comparison.png'))
display(Image('results/accuracy_comparison.png'))

## 11. Kernel fusion evidence

Polygraphy reports the layer count of the optimized engine. ResNet18 ONNX has ~70 nodes; the FP16 TRT engine fuses many of them (typically into ~25–35 layers) — direct evidence of the kernel fusion claim.

In [ ]:
!polygraphy inspect model engines/resnet18.onnx --mode=basic | tee results/onnx_layers.txt | tail -5
!polygraphy inspect model engines/resnet18_fp16.engine --mode=basic | tee results/trt_layers.txt | tail -5

## 12. Headline numbers for the README

If everything ran cleanly you should see something like:
- TRT FP16 P50 latency 35–45% lower than TRT FP32 (batch=1, ResNet18)
- TRT FP16 top-1 within 0.1pp of PyTorch baseline
- TRT INT8 top-1 within ~1pp of PyTorch baseline
- ONNX nodes ≫ TRT layers (kernel fusion)